# A Practical Guide to Quantitative Finance Interviews — Chapter 4 Probability Theory

**Xinfeng Zhou ("the Green Book")** — worked solutions in code.

Third notebook in the set, alongside the Chapter 2 brain teasers
([`quant_finance_interviews.ipynb`](quant_finance_interviews.ipynb)) and the Chapter 3 calculus
notes ([`quant_finance_calculus.ipynb`](quant_finance_calculus.ipynb)). Probability problems
usually *do* carry a parameter to generalise, so — like Chapter 2 — each gets a short
explanation plus **one general-purpose function** (solving for all `n`), with a Monte-Carlo
check where a closed form is subtle.

Covered so far:

| § | Topic |
|---|-------|
| **4.1** | Basic Probability Definitions & Set Operations |
| **4.2** | Combinatorial Analysis |
| **4.3** | Conditional Probability & Bayes' Formula |
| **4.4** | Discrete & Continuous Distributions |
| **4.5** | Expected Value, Variance & Covariance |

## 4.1 Basic Probability Definitions and Set Operations

**The vocabulary.** A random experiment has a **sample space** $\Omega$ (all possible
outcomes); an **event** $A\subseteq\Omega$ is a set of outcomes. A probability measure $P$
assigns each event a number obeying **Kolmogorov's axioms**:
1. $P(A)\ge0$;
2. $P(\Omega)=1$;
3. for mutually exclusive events, $P(A_1\cup A_2\cup\cdots)=\sum_i P(A_i)$.

**Set operations on events** (the algebra of "and / or / not"):
- **Union** $A\cup B$ — $A$ *or* $B$ occurs;
- **Intersection** $A\cap B$ — $A$ *and* $B$ occur;
- **Complement** $A^{c}$ — $A$ does *not* occur, with $P(A^{c})=1-P(A)$;
- **De Morgan:** $(A\cup B)^{c}=A^{c}\cap B^{c}$ and $(A\cap B)^{c}=A^{c}\cup B^{c}$.

**Inclusion–exclusion** repairs the double-counting in a union:
$$P(A\cup B)=P(A)+P(B)-P(A\cap B),$$
$$P(A\cup B\cup C)=P(A)+P(B)+P(C)-P(A\cap B)-P(A\cap C)-P(B\cap C)+P(A\cap B\cap C).$$

**Conditional probability & independence.**
$$P(A\mid B)=\frac{P(A\cap B)}{P(B)}\ \ (P(B)>0),\qquad
A,B\ \text{independent}\iff P(A\cap B)=P(A)\,P(B).$$
Two more workhorses follow: the **law of total probability**
$P(A)=\sum_i P(A\mid B_i)P(B_i)$ over a partition $\{B_i\}$, and **Bayes' rule**
$P(B_i\mid A)=\dfrac{P(A\mid B_i)P(B_i)}{\sum_j P(A\mid B_j)P(B_j)}$.

**Two recurring tricks**, used all over this section:
- **Symmetry** — if two events are interchangeable by relabelling, they carry equal
  probability (so when they also split the remaining "no-tie" cases, each is half of it).
- **Complementary counting** — often $P(A)=1-P(A^{c})$ is far easier than attacking $P(A)$ head-on.

The four problems below are pure applications of these basics.

### 4.1.1 Coin toss game

**Problem.** Gambler $A$ flips $n+1$ fair coins and gambler $B$ flips $n$. What is the
probability that $A$ gets **strictly more heads** than $B$?

**Logic (symmetry — essentially no computation).** Let $H_A,H_B$ be the head counts and
$T_A,T_B$ the tail counts, so $H_A+T_A=n+1$ and $H_B+T_B=n$. Consider
$$X:\ H_A>H_B\ \text{("A has more heads")},\qquad Y:\ T_A>T_B\ \text{("A has more tails")}.$$
- **They are mutually exclusive.** If both held, then $(H_A-H_B)+(T_A-T_B)=(n+1)-n=1$ with
  each difference $\ge1$ — impossible (the two would sum to $\ge2$).
- **One of them always holds.** If neither did, then $H_A\le H_B$ *and* $T_A\le T_B$, forcing
  $n+1=H_A+T_A\le H_B+T_B=n$ — impossible.

So $X$ and $Y$ **partition** the sample space: $P(X)+P(Y)=1$. And since the coins are fair,
swapping heads$\leftrightarrow$tails is a symmetry carrying $X$ to $Y$, so $P(X)=P(Y)$. Hence
$$\boxed{\,P(H_A>H_B)=\tfrac12\,}\qquad\text{for every }n.$$
The function below confirms it by summing the exact binomial probabilities.

In [1]:
from math import comb
from fractions import Fraction

def coin_toss_A_more_heads(n):
    """P(A's heads > B's heads) when A flips n+1 fair coins and B flips n, computed
    exactly from the binomial pmfs. The symmetry argument proves it is 1/2 for every n."""
    total = sum(comb(n + 1, a) * comb(n, b)
                for a in range(n + 2) for b in range(n + 1) if a > b)
    return Fraction(total, 2 ** (2 * n + 1))     # divide by 2^{(n+1)+n}

for n in [0, 1, 2, 5, 10, 25]:
    print(f"n={n:>2}: P(A has more heads) = {coin_toss_A_more_heads(n)}")

n= 0: P(A has more heads) = 1/2
n= 1: P(A has more heads) = 1/2
n= 2: P(A has more heads) = 1/2
n= 5: P(A has more heads) = 1/2
n=10: P(A has more heads) = 1/2
n=25: P(A has more heads) = 1/2


### 4.1.2 Card game

**Problem.** A $52$-card deck has $13$ values ($2,3,\dots,10,J,Q,K,A$), four cards each. You
draw one card and the dealer draws another **without replacement**. You win only if your value
is **strictly higher**; on a tie or a lower value the house wins. What is your winning
probability?

**Logic (symmetry + a single tie term).** Your card and the dealer's are exchangeable, so by
symmetry $P(\text{you}>\text{dealer})=P(\text{dealer}>\text{you})$. Together with a **tie**
these exhaust the outcomes, so
$$P(\text{win})=\frac{1-P(\text{tie})}{2}.$$
Only the tie needs counting: after you draw a card, $51$ remain and $3$ of them share your
value, so $P(\text{tie})=\dfrac{3}{51}=\dfrac1{17}$. Therefore
$$P(\text{win})=\frac{1-\frac1{17}}{2}=\frac{16/17}{2}=\boxed{\dfrac{8}{17}}\approx0.4706,$$
just below a coin flip — the house keeps its edge. In general, for $v$ values with $k$ copies
each ($N=vk$ cards), $P(\text{tie})=\dfrac{k-1}{N-1}$ and
$P(\text{win})=\dfrac12\!\left(1-\dfrac{k-1}{N-1}\right)$.

In [2]:
from fractions import Fraction

def card_win_prob(values=13, copies=4):
    """P(your card's value > dealer's) drawing two cards without replacement from a deck of
    `values` distinct values, `copies` each. P(tie)=(copies-1)/(N-1) and, by symmetry,
    P(win)=(1-P(tie))/2. Returns (P(win), P(tie))."""
    N = values * copies
    p_tie = Fraction(copies - 1, N - 1)
    return (1 - p_tie) / 2, p_tie

w, t = card_win_prob(13, 4)
print(f"standard deck (13 values x 4): P(win) = {w} = {float(w):.4f},  P(tie) = {t}")
for v, k in [(13, 1), (5, 10), (13, 4)]:
    w, t = card_win_prob(v, k)
    print(f"{v:>2} values x {k:>2}: P(win) = {w} = {float(w):.4f}")

standard deck (13 values x 4): P(win) = 8/17 = 0.4706,  P(tie) = 1/17
13 values x  1: P(win) = 1/2 = 0.5000
 5 values x 10: P(win) = 20/49 = 0.4082
13 values x  4: P(win) = 8/17 = 0.4706


### 4.1.3 Drunk passenger

**Problem.** $100$ passengers board in order; passenger $n$ owns seat $n$. The **first**
passenger is drunk and takes a **uniformly random** seat. Everyone after sits in their own seat
if it is free, otherwise in a uniformly random free seat. What is the probability **you**
(passenger $100$) end up in your own seat?

**Logic (a clean symmetry).** Follow the "displaced" passenger — initially the drunk one. Each
time a displaced passenger picks at random, the process ends the moment someone sits in **seat
$1$** (then every later passenger, you included, finds their own seat free → you **win**) or in
**seat $100$** (then you are ultimately bumped → you **lose**). Any random pick that hits some
*other* seat merely hands the "displaced" role to a new passenger and continues. So the outcome
is a race between seat $1$ and seat $100$, and at each random choice those two seats are
**equally likely** to be taken — so by symmetry the process ends on each with probability
$\tfrac12$:
$$\boxed{\,P(\text{you get seat }100)=\tfrac12\,}\qquad(n\ge2).$$
Strikingly this is $\tfrac12$ for **any** number of passengers $\ge2$ — the $100$ is a red
herring. Exact value and a Monte-Carlo check below.

In [3]:
import random
from fractions import Fraction

def drunk_passenger_exact(n):
    """P(passenger n gets their own seat) in the drunk-first-passenger problem:
    exactly 1/2 for n >= 2 (and 1 for n = 1)."""
    return Fraction(1) if n <= 1 else Fraction(1, 2)

def drunk_passenger_sim(n, trials=20000, seed=0):
    """Monte-Carlo estimate of the same probability, boarding the plane `trials` times."""
    rng = random.Random(seed)
    wins = 0
    for _ in range(trials):
        taken = bytearray(n + 1)                       # seats 1..n (index 0 unused)
        taken[rng.randint(1, n)] = 1                    # drunk passenger 1
        for i in range(2, n):                           # sober passengers 2..n-1
            if not taken[i]:
                taken[i] = 1                            # own seat free -> take it
            else:
                free = [s for s in range(1, n + 1) if not taken[s]]
                taken[rng.choice(free)] = 1             # else a random free seat
        wins += (taken[n] == 0)                          # you win iff seat n is still free
    return wins / trials

for n in [2, 5, 100]:
    print(f"n={n:>3}: exact = {drunk_passenger_exact(n)}   simulated = {drunk_passenger_sim(n):.3f}")

n=  2: exact = 1/2   simulated = 0.503
n=  5: exact = 1/2   simulated = 0.495
n=100: exact = 1/2   simulated = 0.503


### 4.1.4 N points on a circle

**Problem.** $N$ points are dropped uniformly at random on a circle's circumference. What is the
probability they **all lie within some semicircle**?

**Logic (fix a point, then sum disjoint cases).** For each point $i$, let $E_i$ be the event
"starting at point $i$ and sweeping **clockwise**, the next half-circle contains all the other
$N-1$ points." Each other point independently falls in that fixed semicircle with probability
$\tfrac12$, so
$$P(E_i)=\left(\tfrac12\right)^{N-1}.$$
The events $E_1,\dots,E_N$ are **mutually exclusive** — at most one point can be the clockwise
"first" point of an all-containing semicircle. Since "all in some semicircle" is exactly "some
$E_i$ occurs,"
$$P(\text{all within a semicircle})=\sum_{i=1}^{N}P(E_i)=\boxed{\dfrac{N}{2^{\,N-1}}}.$$
Sanity: $N=2\Rightarrow1$ (two points always fit), $N=3\Rightarrow\tfrac34$,
$N=4\Rightarrow\tfrac12$. The function returns the exact value; the simulation checks it via the
equivalent test *"the largest gap between adjacent points is at least half the circle."*

In [4]:
import random
from fractions import Fraction

def semicircle_prob(N):
    """P(all N uniformly-random points on a circle lie within some semicircle) = N / 2^{N-1}."""
    return Fraction(N, 2 ** (N - 1))

def semicircle_sim(N, trials=200000, seed=0):
    """Monte-Carlo check: all points fit in a semicircle iff some gap between adjacent points
    (around the circle) is at least half the circumference."""
    rng = random.Random(seed)
    hits = 0
    for _ in range(trials):
        pts = sorted(rng.random() for _ in range(N))        # positions as fractions of the circle
        gaps = [pts[i + 1] - pts[i] for i in range(N - 1)] + [1 - pts[-1] + pts[0]]
        hits += (max(gaps) >= 0.5)
    return hits / trials

for N in [2, 3, 4, 5, 6]:
    exact = semicircle_prob(N)
    print(f"N={N}: exact = {exact} = {float(exact):.4f}   simulated = {semicircle_sim(N):.4f}")

N=2: exact = 1 = 1.0000   simulated = 1.0000
N=3: exact = 3/4 = 0.7500   simulated = 0.7506
N=4: exact = 1/2 = 0.5000   simulated = 0.5011
N=5: exact = 5/16 = 0.3125   simulated = 0.3131
N=6: exact = 3/16 = 0.1875   simulated = 0.1882


## 4.2 Combinatorial Analysis

Counting is the engine of discrete probability: most "what's the chance" questions reduce to
*(favourable arrangements) / (total arrangements)*. Five tools do almost all the work.

**Basic principle of counting (multiplication rule).** If a task is carried out in $k$
independent stages offering $n_1,n_2,\dots,n_k$ choices, the number of overall outcomes is
$$n_1\times n_2\times\cdots\times n_k .$$
(For *mutually exclusive alternatives* you **add** rather than multiply.)

**Permutations — ordered arrangements.** The number of ways to arrange $r$ of $n$ distinct
objects, where **order matters**, is
$$P(n,r)=\frac{n!}{(n-r)!};\qquad\text{all }n\text{ objects: }P(n,n)=n! .$$

**Combinations — unordered selections.** The number of ways to choose $r$ of $n$ objects, where
**order does not matter**, is
$$\binom{n}{r}=\frac{n!}{r!\,(n-r)!}=\frac{P(n,r)}{r!},\qquad \binom{n}{r}=\binom{n}{n-r}.$$

**Binomial theorem.** Those combinations are exactly the coefficients of
$$(x+y)^{n}=\sum_{k=0}^{n}\binom{n}{k}x^{k}y^{n-k},$$
so putting $x=y=1$ gives $\displaystyle\sum_{k=0}^{n}\binom{n}{k}=2^{n}$ — the number of subsets
of an $n$-element set.

**Inclusion–exclusion principle.** To count a union without double-counting overlaps,
$$\Big|\bigcup_{i=1}^{n}A_i\Big|=\sum_i|A_i|-\sum_{i<j}|A_i\cap A_j|+\sum_{i<j<k}|A_i\cap A_j\cap A_k|-\cdots+(-1)^{\,n+1}\Big|\bigcap_{i=1}^{n}A_i\Big| .$$
(The derangement count in 4.2.5 is a textbook application.)

The five problems below apply these directly.

### 4.2.1 Poker hands

**Problem.** From a standard $52$-card deck ($13$ values $\times$ $4$ suits), a poker hand is
$5$ cards. Find the probability of **four-of-a-kind**, a **full house** (three of one value and
two of another), and **two pairs**.

**Counting.** Every hand is equally likely, so each probability is
(favourable hands)$/\binom{52}{5}$ with $\binom{52}{5}=2{,}598{,}960$. Build each favourable hand
by the multiplication rule:

- **Four-of-a-kind:** pick the quad's value ($13$), take all $4$ suits ($\binom{4}{4}=1$), then
  any $5$th card from the other $48$: $\;13\cdot1\cdot48=624$.
- **Full house:** pick the triple's value ($13$) and $3$ of its suits ($\binom{4}{3}=4$), then the
  pair's value ($12$) and $2$ suits ($\binom{4}{2}=6$): $\;13\cdot4\cdot12\cdot6=3{,}744$.
- **Two pairs:** pick the two pair-values ($\binom{13}{2}=78$), $2$ suits for each
  ($\binom{4}{2}^{2}=36$), then a $5$th card of a different value ($11\cdot4=44$):
  $\;78\cdot36\cdot44=123{,}552$.

So $P(\text{four})\approx0.00024$, $P(\text{full house})\approx0.00144$,
$P(\text{two pairs})\approx0.0475$. The function computes these, generalised to any
values$\times$suits deck.

In [5]:
from math import comb

def poker_probabilities(values=13, suits=4):
    """Counts of four-of-a-kind, full house, and two pairs in a 5-card hand from a deck of
    `values` distinct values x `suits` suits; each probability is count / C(values*suits, 5).
    All counts come from the multiplication rule."""
    total = comb(values * suits, 5)
    four = values * comb(suits, 4) * (values - 1) * suits
    full = values * comb(suits, 3) * (values - 1) * comb(suits, 2)
    two_pair = comb(values, 2) * comb(suits, 2) ** 2 * (values - 2) * suits
    return {"four_of_a_kind": four, "full_house": full, "two_pairs": two_pair, "total": total}

r = poker_probabilities()
for hand in ("four_of_a_kind", "full_house", "two_pairs"):
    c = r[hand]
    print(f"{hand:>15}: {c:>7,} / {r['total']:,} = {c/r['total']:.6f}  (~1 in {r['total']/c:,.0f})")

 four_of_a_kind:     624 / 2,598,960 = 0.000240  (~1 in 4,165)
     full_house:   3,744 / 2,598,960 = 0.001441  (~1 in 694)
      two_pairs: 123,552 / 2,598,960 = 0.047539  (~1 in 21)


### 4.2.2 Hopping rabbit

**Problem.** A rabbit at the bottom of a staircase of $n$ stairs hops up $1$ or $2$ stairs at a
time. How many distinct ways can it reach the top?

**Logic (recursion → Fibonacci).** Let $f(n)$ be the number of ways. The rabbit's **last** hop
lands on stair $n$ from either stair $n-1$ (a $1$-hop) or stair $n-2$ (a $2$-hop), and those two
sets of routes are disjoint, so
$$f(n)=f(n-1)+f(n-2),\qquad f(1)=1,\ f(2)=2.$$
That is the **Fibonacci** recurrence: $f(n)=1,2,3,5,8,13,\dots$ (indeed $f(n)=F_{n+1}$). The
function returns $f(n)$ in $O(n)$.

In [6]:
def rabbit_ways(n):
    """Ways to climb n stairs in 1- or 2-hops. f(n)=f(n-1)+f(n-2) (the last hop comes from stair
    n-1 or n-2), with f(0)=f(1)=1 -> a Fibonacci sequence."""
    a, b = 1, 1
    for _ in range(n):
        a, b = b, a + b
    return a

print("stairs n :", list(range(1, 11)))
print("ways f(n):", [rabbit_ways(n) for n in range(1, 11)])

stairs n : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
ways f(n): [1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### 4.2.3 Screwy pirates 2

**Problem.** $11$ pirates lock the loot in a safe that opens only when **any majority**
($\ge6$) work together. Each lock can carry many keys, each key opens one lock, and a pirate may
hold several keys. What is the **smallest number of locks**, and how many **keys per pirate**?

**Logic (block every minority).** The requirement is: every group of $6$ opens *all* locks, yet
every group of $5$ is stopped by *some* lock. So dedicate one lock to each minority of size $5$
and give that lock's keys only to the **other $6$** pirates.

- A group of $5$ is exactly one such minority $S$; the lock assigned to $S$ is held only by the
  $6$ outsiders, so this group lacks it — blocked. ✓
- A group of $6$: for *any* lock (assigned to some $5$-set $S$), the six include at least one
  pirate outside $S$, who holds that key — so they open everything. ✓

Hence **one lock per $5$-subset**: $\binom{11}{5}=462$ locks. A given pirate holds a lock's key
iff he is *not* in its $5$-subset, i.e. the subset is chosen from the other $10$:
$\binom{10}{5}=252$ keys each. In general, for $n$ pirates with majority threshold $m$ (minority
$m-1$): $\binom{n}{m-1}$ locks and $\binom{n-1}{m-1}$ keys per pirate.

In [7]:
from math import comb

def screwy_pirates2(n=11, majority=6):
    """Smallest number of locks and keys-per-pirate so any `majority` pirates can open the safe
    but no smaller group can. One lock per minority of size majority-1, its keys held by everyone
    outside that minority. Returns (locks, keys_per_pirate)."""
    minority = majority - 1
    return comb(n, minority), comb(n - 1, minority)

print("11 pirates, majority 6 ->", screwy_pirates2(11, 6), "(locks, keys per pirate)")
for n, m in [(3, 2), (5, 3), (11, 6)]:
    print(f"  n={n:>2}, majority={m}: {screwy_pirates2(n, m)}")

11 pirates, majority 6 -> (462, 252) (locks, keys per pirate)
  n= 3, majority=2: (3, 2)
  n= 5, majority=3: (10, 6)
  n=11, majority=6: (462, 252)


### 4.2.4 Chess tournament

**Problem.** $2^{n}$ players with strictly ranked skills play a knockout tournament; opponents
are drawn **at random** each round (except the final), and the stronger player always wins. What
is the probability that the top two, players $1$ and $2$, meet in the **final**?

**Logic.** Player $1$ beats everyone, so player $1$ reaches the final with certainty. Players $1$
and $2$ therefore meet in the final **iff player $2$ survives every earlier round**, i.e. iff
player $2$ is never drawn against player $1$ before the final. In a round beginning with $m$
players, player $2$ faces a uniformly random one of the other $m-1$, avoiding player $1$ with
probability $\tfrac{m-2}{m-1}$. Multiplying over the pre-final rounds ($m=2^{n},2^{n-1},\dots,4$),
$$P=\prod_{m\in\{4,8,\dots,2^{n}\}}\frac{m-2}{m-1}=\boxed{\dfrac{2^{\,n-1}}{2^{\,n}-1}} .$$
(Check: $n=2\Rightarrow\tfrac23$, $n=3\Rightarrow\tfrac47$; as $n$ grows it tends to $\tfrac12$.)

In [8]:
from fractions import Fraction

def chess_final_prob(n):
    """P(the two best of 2^n players meet in the final of a random knockout). Player 1 always
    wins; player 2 must dodge player 1 in each round before the final. The product telescopes
    to 2^{n-1}/(2^n - 1)."""
    prob, m = Fraction(1), 2 ** n
    while m > 2:
        prob *= Fraction(m - 2, m - 1)        # player 2 avoids player 1 this round
        m //= 2
    return prob

for n in range(1, 7):
    p = chess_final_prob(n)
    print(f"n={n}: {2**n:>2} players -> P = {p} = {float(p):.4f}   [2^(n-1)/(2^n-1)]")

n=1:  2 players -> P = 1 = 1.0000   [2^(n-1)/(2^n-1)]
n=2:  4 players -> P = 2/3 = 0.6667   [2^(n-1)/(2^n-1)]
n=3:  8 players -> P = 4/7 = 0.5714   [2^(n-1)/(2^n-1)]
n=4: 16 players -> P = 8/15 = 0.5333   [2^(n-1)/(2^n-1)]
n=5: 32 players -> P = 16/31 = 0.5161   [2^(n-1)/(2^n-1)]
n=6: 64 players -> P = 32/63 = 0.5079   [2^(n-1)/(2^n-1)]


### 4.2.5 Application letters

**Problem.** Five personalised cover letters are stuffed into five addressed envelopes **at
random**. What is the probability that **all five** go to the wrong firm?

**Logic (derangements via inclusion–exclusion).** "All wrong" means the random permutation of
letters leaves **no letter in its correct envelope** — a *derangement*. Let $A_i$ be the event
that letter $i$ *is* correctly placed; inclusion–exclusion on $\bigcup A_i$ (at least one
correct) leaves, for the complement (none correct),
$$P(\text{all wrong})=\frac{D_n}{n!}=\sum_{k=0}^{n}\frac{(-1)^{k}}{k!},$$
where $D_n$ is the number of derangements. For $n=5$,
$$P=1-1+\tfrac12-\tfrac16+\tfrac1{24}-\tfrac1{120}=\frac{44}{120}=\boxed{\frac{11}{30}}\approx0.3667 .$$
As $n\to\infty$ this converges rapidly to $1/e\approx0.3679$ — it is essentially $1/e$ already at
$n=5$.

In [9]:
from math import factorial
from fractions import Fraction

def derangement_prob(n):
    """P(a random permutation of n items leaves nothing in its own place) = D_n/n!
    = sum_{k=0}^n (-1)^k/k!  ->  1/e as n grows."""
    return sum(Fraction((-1) ** k, factorial(k)) for k in range(n + 1))

for n in [3, 4, 5, 6, 10]:
    p = derangement_prob(n)
    print(f"n={n:>2}: P(all wrong) = {p} = {float(p):.5f}   (D_n = {round(factorial(n) * float(p))})")
print(f"limit 1/e = {1 / 2.718281828459045:.5f}")

n= 3: P(all wrong) = 1/3 = 0.33333   (D_n = 2)
n= 4: P(all wrong) = 3/8 = 0.37500   (D_n = 9)
n= 5: P(all wrong) = 11/30 = 0.36667   (D_n = 44)
n= 6: P(all wrong) = 53/144 = 0.36806   (D_n = 265)
n=10: P(all wrong) = 16481/44800 = 0.36788   (D_n = 1334961)
limit 1/e = 0.36788


### 4.2.6 Birthday problem

**Problem.** In a class of $n$ people (birthdays equally likely among $365$ days), how large must
$n$ be for the probability that **some two share a birthday** to exceed $\tfrac12$?

**Logic (complementary counting).** Attack the complement — *all birthdays distinct*. Lining the
people up, the first can be any day, the second must avoid $1$ day, the third $2$, and so on:
$$P(\text{all distinct})=\frac{365}{365}\cdot\frac{364}{365}\cdots\frac{365-n+1}{365}
=\frac{365!}{(365-n)!\,365^{\,n}},$$
so $P(\text{some match})=1-P(\text{all distinct})$. This crosses $\tfrac12$ at $\boxed{n=23}$
($P\approx0.507$, while $n=22$ gives only $\approx0.476$) — famously small, because $n$ people form
$\binom{n}{2}$ pairs that could collide, and $\binom{23}{2}=253$. The function finds the threshold
for any number of days.

In [10]:
from fractions import Fraction

def birthday_prob(n, days=365):
    """P(at least two of n people share a birthday) = 1 - prod_{k=0}^{n-1} (days-k)/days."""
    p_distinct = Fraction(1)
    for k in range(n):
        p_distinct *= Fraction(days - k, days)
    return 1 - p_distinct

def birthday_min_people(days=365, threshold=Fraction(1, 2)):
    """Smallest class size whose shared-birthday probability exceeds `threshold`."""
    n = 1
    while birthday_prob(n, days) <= threshold:
        n += 1
    return n

print("days=365: need", birthday_min_people(), "people for P(shared birthday) > 1/2")
for n in [10, 22, 23, 50, 70]:
    print(f"  n={n:>2}: P(some share) = {float(birthday_prob(n)):.4f}")

days=365: need 23 people for P(shared birthday) > 1/2
  n=10: P(some share) = 0.1169
  n=22: P(some share) = 0.4757
  n=23: P(some share) = 0.5073
  n=50: P(some share) = 0.9704
  n=70: P(some share) = 0.9992


### 4.2.7 100th digit

**Problem.** What is the $100$th digit to the **right** of the decimal point of $(1+\sqrt2)^{3000}$?

**Logic (the conjugate cancels the irrational part).** The conjugate $1-\sqrt2$ is the other root of
$x^{2}-2x-1=0$, so the sum
$$a_n=(1+\sqrt2)^{\,n}+(1-\sqrt2)^{\,n}$$
is always an **integer** (the $\sqrt2$ terms cancel), obeying $a_n=2a_{n-1}+a_{n-2}$, $a_0=a_1=2$.
Hence
$$(1+\sqrt2)^{3000}=a_{3000}-(1-\sqrt2)^{3000}=a_{3000}-(\sqrt2-1)^{3000},$$
using $(1-\sqrt2)^{3000}=(\sqrt2-1)^{3000}$ (even exponent). Now $\sqrt2-1=\dfrac{1}{\sqrt2+1}<\dfrac12$,
so
$$0<(\sqrt2-1)^{3000}<\left(\tfrac12\right)^{3000}<10^{-100}\qquad(\text{indeed }2^{3000}\text{ has }904\text{ digits}).$$
So $(1+\sqrt2)^{3000}$ is an integer **minus** a positive amount below $10^{-100}$: its fractional part
is $1-(\sqrt2-1)^{3000}=0.\underbrace{99\cdots9}_{\ge100}\ldots$, and **every one of the first $100$
decimal digits is $9$** — in particular the $100$th digit is $\boxed{9}$.

In [11]:
# a_n = (1+sqrt2)^n + (1-sqrt2)^n is an integer: a_n = 2 a_{n-1} + a_{n-2}, a_0=a_1=2
a, b = 2, 2
for _ in range(2, 3001):
    a, b = b, 2 * b + a
a_3000 = b

# (sqrt2-1)^3000 < (1/2)^3000 < 10^-100, so the fractional part of (1+sqrt2)^3000 is
# 1 - tiny = 0.999...9, and its k-th decimal digit is 9 for every k up to ~900.
print("2^3000 has", len(str(2 ** 3000)), "digits => (sqrt2-1)^3000 < 10^-100:", 2 ** 3000 > 10 ** 100)
digit_100 = (a_3000 * 10 ** 100 - 1) % 10       # last digit of floor((1+sqrt2)^3000 * 10^100)
print("100th digit to the right of the decimal point:", digit_100)

2^3000 has 904 digits => (sqrt2-1)^3000 < 10^-100: True
100th digit to the right of the decimal point: 9


### 4.2.8 Cubic of integer

**Problem.** For an integer $x$ chosen uniformly in $[1,10^{12}]$, what is the probability that
$x^{3}$ **ends in $11$** (its last two digits are $11$)?

**Logic (periodicity mod $100$).** "Ends in $11$" means $x^{3}\equiv11\pmod{100}$, and the last two
digits of $x^{3}$ depend only on $x\bmod100$ — so the pattern repeats every $100$ integers. Because
$\gcd\big(3,\varphi(100)\big)=\gcd(3,40)=1$, cubing is a **bijection** on the units mod $100$, so
$x^{3}\equiv11$ has exactly **one** solution; solving (e.g. by CRT on mod $4$ and mod $25$) gives
$x\equiv71\pmod{100}$ — check $71^{3}=357{,}911$. Since $10^{12}$ is a whole number of $100$-blocks,
each residue occurs equally often, so
$$P\big(x^{3}\text{ ends in }11\big)=\frac{1}{100}=\boxed{0.01}.$$
The function counts the qualifying residues for any suffix and modulus.

In [12]:
from fractions import Fraction

def cube_ending_prob(suffix=11, digits=2, upper=10 ** 12):
    """P(x^3 ends with `suffix` in its last `digits` digits) for x uniform in 1..upper. Those last
    digits depend only on x mod 10^digits, so when upper is a whole number of periods the answer is
    (# residues r with r^3 == suffix mod 10^digits) / 10^digits. Returns (prob, residues)."""
    mod = 10 ** digits
    residues = [r for r in range(mod) if pow(r, 3, mod) == suffix % mod]
    assert upper % mod == 0, "upper must be a whole number of 10^digits blocks"
    return Fraction(len(residues), mod), residues

p, res = cube_ending_prob(11, 2)
print(f"x^3 ends in 11: P = {p} = {float(p)},  residues mod 100 = {res}  (check 71^3 = {71 ** 3})")

x^3 ends in 11: P = 1/100 = 0.01,  residues mod 100 = [71]  (check 71^3 = 357911)


## 4.3 Conditional Probability and Bayes' Formula

**Conditional probability $P(A\mid B)$.** The probability of $A$ *given that* $B$ has happened,
rescaling the world to $B$:
$$P(A\mid B)=\frac{P(A\cap B)}{P(B)}\qquad(P(B)>0).$$
*Example:* for a fair die, $P(\text{6}\mid\text{even})=\dfrac{1/6}{1/2}=\dfrac13$ — knowing it is even
leaves three equally likely faces.

**Multiplication rule.** Rearranging the definition builds joint probabilities from conditional ones:
$$P(A\cap B)=P(A\mid B)\,P(B)=P(B\mid A)\,P(A),$$
and in a chain $P(A_1\cap\cdots\cap A_n)=P(A_1)\,P(A_2\mid A_1)\cdots P(A_n\mid A_1\cap\cdots\cap A_{n-1})$.
*Example:* two aces off the top of a deck, $P=\dfrac{4}{52}\cdot\dfrac{3}{51}$.

**Law of total probability.** If $\{B_1,\dots,B_k\}$ partition the sample space, any $A$ is the sum of
its pieces:
$$P(A)=\sum_{i=1}^{k}P(A\mid B_i)\,P(B_i).$$
*Example:* pick one of two urns at random then draw a ball —
$P(\text{red})=\tfrac12P(\text{red}\mid U_1)+\tfrac12P(\text{red}\mid U_2)$.

**Independent events.** $A$ and $B$ are independent when one carries no information about the other:
$$A\perp B \iff P(A\cap B)=P(A)\,P(B) \iff P(A\mid B)=P(A).$$
*Example:* two separate fair tosses, $P(\text{both heads})=\tfrac12\cdot\tfrac12=\tfrac14$.

**Bayes' formula.** Invert a conditional — turn $P(B\mid A)$ into $P(A\mid B)$ — by combining the
multiplication rule with total probability:
$$P(A\mid B)=\frac{P(B\mid A)\,P(A)}{P(B)}=\frac{P(B\mid A)\,P(A)}{P(B\mid A)P(A)+P(B\mid A^{c})P(A^{c})}.$$
It updates a **prior** $P(A)$ into a **posterior** $P(A\mid B)$ after seeing evidence $B$ — the engine
behind the "unfair coin" and Russian-roulette problems below.

### 4.3.1 Boys and girls

**Problem.** *Part A:* Ms. Jackson has two children and at least one is a boy. What is the probability
that **both** are boys? *Part B:* You meet Ms. Parker walking with one of her two children, and that
child is a boy. Now what is the probability both are boys?

**Logic — the same-sounding questions condition on different events.** With two children the equally
likely sample space is $\{BB,BG,GB,GG\}$.
- **Part A** conditions on *"at least one boy"*, which rules out only $GG$, leaving $\{BB,BG,GB\}$, of
  which one is $BB$:
  $$P(BB\mid\text{at least one boy})=\frac{P(BB)}{P(\text{at least one boy})}=\frac{1/4}{3/4}=\boxed{\tfrac13}.$$
- **Part B** conditions on *a specific, observed child being a boy*. That child is settled; the
  **other** is an independent fair coin, so $P(\text{both boys}\mid\text{this child is a boy})=\boxed{\tfrac12}.$

The subtlety: "at least one of the two is a boy" (A) is *weaker* information than "this particular child
is a boy" (B), so it leaves more uncertainty and a smaller answer. For $k$ children, Part A gives
$\dfrac{1}{2^{k}-1}$ and Part B gives $\dfrac{1}{2^{k-1}}$.

In [13]:
from itertools import product
from fractions import Fraction

def boys_partA(k=2):
    """P(all boys | at least one boy) among k children = 1/(2^k - 1)."""
    outcomes = list(product("BG", repeat=k))
    at_least_one_boy = [o for o in outcomes if "B" in o]
    return Fraction(sum(set(o) == {"B"} for o in at_least_one_boy), len(at_least_one_boy))

def boys_partB(k=2):
    """P(all boys | one specific observed child is a boy) = 1/2^{k-1}
    (the other k-1 children are independent fair coins)."""
    return Fraction(1, 2 ** (k - 1))

for k in [2, 3, 4]:
    print(f"k={k}: Part A P(all boys | >=1 boy) = {boys_partA(k)};   "
          f"Part B P(all boys | saw a boy) = {boys_partB(k)}")

k=2: Part A P(all boys | >=1 boy) = 1/3;   Part B P(all boys | saw a boy) = 1/2
k=3: Part A P(all boys | >=1 boy) = 1/7;   Part B P(all boys | saw a boy) = 1/4
k=4: Part A P(all boys | >=1 boy) = 1/15;   Part B P(all boys | saw a boy) = 1/8


### 4.3.2 All-girl world?

**Problem.** Every couple keeps having children until they get a girl, then stops. Each child is a girl
with probability $\tfrac12$, independently. In the long run, what fraction of the population is female?

**Logic (the stopping rule is a red herring).** Sex is decided **independently** at each birth with
probability $\tfrac12$ — no rule about *when to stop* can change that coin. Every child ever born, in
any family, is a girl with probability $\tfrac12$, so the fraction of girls stays $\boxed{50\%}$.
Concretely each family ends with **exactly one girl** after, on average, $\tfrac{1}{1/2}=2$ children (a
geometric count), giving an expected girl-fraction of $\tfrac12$ per family — matching the population.
The word "prefer" misleads; independence is all that matters.

In [14]:
import random

def all_girl_world_sim(families=200000, p_girl=0.5, seed=0):
    """Simulate the 'have children until a girl, then stop' rule and report the girl fraction.
    It is ~p_girl regardless of the stopping rule."""
    rng = random.Random(seed)
    girls = children = 0
    for _ in range(families):
        while True:
            children += 1
            if rng.random() < p_girl:
                girls += 1          # a girl -> the family stops
                break
    return girls / children

print(f"simulated girl fraction = {all_girl_world_sim():.4f}   (theory 0.5)")

simulated girl fraction = 0.5005   (theory 0.5)


### 4.3.3 Unfair coin

**Problem.** Among $1000$ coins one is two-headed and the other $999$ are fair. You pick one at random
and toss it $10$ times — all heads. What is the probability you picked the two-headed coin?

**Logic (Bayes' formula).** Let $A$ = "picked the unfair coin", $B$ = "$10$ heads in a row". Priors
$P(A)=\tfrac1{1000}$, $P(A^{c})=\tfrac{999}{1000}$; likelihoods $P(B\mid A)=1$ and
$P(B\mid A^{c})=(\tfrac12)^{10}=\tfrac1{1024}$. Then
$$P(A\mid B)=\frac{1\cdot\frac1{1000}}{1\cdot\frac1{1000}+\frac1{1024}\cdot\frac{999}{1000}}
=\frac{1024}{2023}\approx\boxed{0.506}.$$
Ten heads is strong evidence, yet the prior was so small ($1/1000$) that the posterior is only about a
coin flip. In general, for $N$ coins and $k$ heads, $P(A\mid B)=\dfrac{2^{k}}{2^{k}+N-1}$.

In [15]:
from fractions import Fraction

def unfair_coin_posterior(n_coins=1000, heads=10):
    """P(the chosen coin is the two-headed one | `heads` heads in a row) among n_coins (one unfair).
    Bayes gives 2^heads / (2^heads + n_coins - 1)."""
    return Fraction(2 ** heads, 2 ** heads + n_coins - 1)

for n, k in [(1000, 10), (1000, 20), (100, 10)]:
    p = unfair_coin_posterior(n, k)
    print(f"{n} coins, {k} heads: P(unfair) = {p} = {float(p):.4f}")

1000 coins, 10 heads: P(unfair) = 1024/2023 = 0.5062
1000 coins, 20 heads: P(unfair) = 1048576/1049575 = 0.9990
100 coins, 10 heads: P(unfair) = 1024/1123 = 0.9118


### 4.3.4 Fair probability from an unfair coin

**Problem.** A coin is biased toward heads or tails by an **unknown** amount. Can you still simulate a
fair $50/50$ decision with it?

**Logic (von Neumann's trick — pair the tosses).** One toss cannot do it, but two can. Let $p_H,p_T$
($p_H+p_T=1$) be the unknown probabilities. Toss **twice**; the two *mixed* outcomes are equally likely
whatever the bias:
$$P(HT)=p_H p_T=p_T p_H=P(TH).$$
So call **$HT$ a "win" and $TH$ a "loss"**, and **discard $HH$ and $TT$** (re-toss the pair). Since $HT$
and $TH$ are equally likely, the decision is exactly fair — for any bias except the degenerate
$p_H\in\{0,1\}$. (The expected number of pairs needed is $\tfrac{1}{2p_Hp_T}$, largest for very biased
coins.)

In [16]:
import random

def von_neumann_fair(p_head, trials=200000, seed=0):
    """Von Neumann extractor: toss the biased coin in pairs; HT -> 1, TH -> 0, discard HH/TT. Returns
    the empirical fraction of 1s, which is ~0.5 for any 0 < p_head < 1."""
    rng = random.Random(seed)
    ones = total = 0
    for _ in range(trials):
        a = rng.random() < p_head
        b = rng.random() < p_head
        if a and not b:        # HT -> win
            ones += 1; total += 1
        elif b and not a:      # TH -> loss
            total += 1
    return ones / total

for p in [0.5, 0.8, 0.1]:
    print(f"biased p(head)={p}: extracted fair-bit fraction = {von_neumann_fair(p):.4f}")

biased p(head)=0.5: extracted fair-bit fraction = 0.5023
biased p(head)=0.8: extracted fair-bit fraction = 0.5007
biased p(head)=0.1: extracted fair-bit fraction = 0.5035


### 4.3.5 Dart game

**Problem.** Jason throws darts at a bullseye with **constant** skill, so all throws are exchangeable.
His second dart lands farther from the center than his first; if he throws a third, what is the
probability it too lands farther than the first? More generally, given the first dart is the closest of
$n$ throws, what is the probability the $(n{+}1)$th lands farther than the first?

**Logic (symmetry of ranks).** With constant skill the throws are exchangeable, so every ordering of
distances is equally likely. The key is whether the $(n{+}1)$th dart is the **overall best** (closest):
by exchangeability $P\big((n{+}1)\text{th is best of all }n{+}1\big)=\tfrac1{n+1}$, and this is
**independent** of how the first $n$ were ordered among themselves. Given the first dart is the best of
the first $n$, the $(n{+}1)$th is farther than the first **iff** it is not the new overall best, so
$$P\big((n{+}1)\text{th farther than the first}\big)=1-\frac1{n+1}=\boxed{\frac{n}{n+1}}.$$
For the three-dart question ($n=2$: the first dart is the closer of the first two), this is $\tfrac23$.

In [17]:
from fractions import Fraction
from itertools import permutations

def dart_prob(n):
    """Given the 1st of n exchangeable throws is the closest of those n, the probability that the
    (n+1)th lands farther from the center than the 1st is n/(n+1)."""
    return Fraction(n, n + 1)

def dart_prob_bruteforce(n):
    """Check by enumerating all orderings of n+1 distinct distances (perm[i] = rank of dart i, 0 =
    closest): among orderings where dart 1 is the best of darts 1..n, the fraction where dart n+1 is
    farther than dart 1."""
    cond = fav = 0
    for perm in permutations(range(n + 1)):
        if perm[0] == min(perm[:n]):
            cond += 1
            fav += (perm[n] > perm[0])
    return Fraction(fav, cond)

for n in [2, 3, 4, 5]:
    print(f"n={n}: n/(n+1) = {dart_prob(n)},  brute force = {dart_prob_bruteforce(n)}")

n=2: n/(n+1) = 2/3,  brute force = 2/3
n=3: n/(n+1) = 3/4,  brute force = 3/4
n=4: n/(n+1) = 4/5,  brute force = 4/5
n=5: n/(n+1) = 5/6,  brute force = 5/6


### 4.3.6 Russian Roulette Series

A $6$-chamber revolver; two players alternate pulling the trigger on themselves, and whoever meets the
live round loses. Each part is a conditional-probability decision.

**Part 1 — one bullet, *no* re-spin; go first or second?** The bullet sits in a uniformly random chamber
$1,\dots,6$, fired in order. The first player pulls chambers $1,3,5$ and the second pulls $2,4,6$, so
each loses with probability $\tfrac{3}{6}=\boxed{\tfrac12}$ — **it makes no difference.**

**Part 2 — one bullet, re-spin after *every* pull; go first or second?** Each pull is now an independent
$\tfrac16$ chance of losing. The first player faces the gun on turns $1,3,5,\dots$, so
$$P(\text{first loses})=\sum_{k\ge0}\Big(\tfrac56\Big)^{2k}\tfrac16=\frac{1/6}{1-(5/6)^{2}}=\frac{6}{11},$$
and the second loses with probability $\tfrac{5}{11}$. **Go second** ($\tfrac5{11}<\tfrac6{11}$).

**Part 3 — two bullets placed at random; the opponent went first and survived. Re-spin before your
pull?** *Without* a spin you pull chamber $2$ knowing chamber $1$ was empty, so the two bullets are
uniform among chambers $2\text{–}6$ and $P(\text{you lose})=\tfrac{2}{5}$. *With* a spin your chamber is
fresh: $P=\tfrac{2}{6}=\tfrac13$. Since $\tfrac13<\tfrac25$, **spin the barrel.**

**Part 4 — two bullets in *consecutive* chambers; the opponent survived the first pull. Re-spin?** The
bullets form one of $6$ adjacent (cyclic) pairs. Chamber $1$ being empty rules out the pairs $(6,1)$ and
$(1,2)$, leaving $(2,3),(3,4),(4,5),(5,6)$ — only the first of which contains chamber $2$. So *without* a
spin $P(\text{you lose})=\tfrac14$, versus $\tfrac13$ *with* a spin. Since $\tfrac14<\tfrac13$, **do not
spin.**

The contrast between Parts 3 and 4 is the lesson: surviving chamber $1$ is *good news* about chamber $2$
only when the bullets are consecutive (it pushes the pair away from the start), so there you keep the
barrel; with scattered bullets, surviving instead *concentrates* the two into fewer chambers, so you
spin.

In [18]:
from itertools import combinations
from fractions import Fraction

# Part 1: one bullet, no spin -> first player pulls chambers 1,3,5
p1_first = Fraction(sum(c % 2 == 1 for c in range(1, 7)), 6)

# Part 2: one bullet, re-spin each pull -> geometric race
p = Fraction(1, 6)
p2_first = p / (1 - (1 - p) ** 2)

# Part 3: two random bullets; chamber 1 known empty -> you pull chamber 2
c1_empty = [b for b in combinations(range(1, 7), 2) if 1 not in b]
p3_nospin = Fraction(sum(2 in b for b in c1_empty), len(c1_empty))
p3_spin = Fraction(2, 6)

# Part 4: two consecutive (cyclic) bullets; chamber 1 empty -> you pull chamber 2
pairs = [tuple(sorted(((i % 6) + 1, ((i + 1) % 6) + 1))) for i in range(6)]
pool = [b for b in pairs if 1 not in b]
p4_nospin = Fraction(sum(2 in b for b in pool), len(pool))
p4_spin = Fraction(2, 6)

print(f"Part 1: P(first loses) = {p1_first} = P(second)  -> no advantage")
print(f"Part 2: P(first) = {p2_first}, P(second) = {1 - p2_first}  -> go second")
print(f"Part 3: no-spin = {p3_nospin}, spin = {p3_spin}  -> {'spin' if p3_spin < p3_nospin else 'do not spin'}")
print(f"Part 4: no-spin = {p4_nospin}, spin = {p4_spin}  -> {'do not spin' if p4_nospin < p4_spin else 'spin'}")

Part 1: P(first loses) = 1/2 = P(second)  -> no advantage
Part 2: P(first) = 6/11, P(second) = 5/11  -> go second
Part 3: no-spin = 2/5, spin = 1/3  -> spin
Part 4: no-spin = 1/4, spin = 1/3  -> do not spin


### 4.3.7 Birthday line

**Problem.** People line up with independent, uniform birthdays over $365$ days. The manager gives a free
ticket to the **first person whose birthday matches someone already ahead** of them in line. You may pick
any position; which position $n$ maximises your chance of winning?

**Logic.** To win from position $n$, the first $n-1$ people must all have **distinct** birthdays (else
someone ahead wins first) *and* your birthday must match one of those $n-1$:
$$p(n)=\underbrace{\frac{365\cdot364\cdots(365-n+2)}{365^{\,n-1}}}_{\text{first }n-1\text{ all distinct}}\times\underbrace{\frac{n-1}{365}}_{\text{you match one of them}}.$$
As $n$ grows the second factor rises while the first falls; the product peaks where $p(n)\ge p(n-1)$ and
$p(n)\ge p(n+1)$. Those reduce to $n^{2}-3n-363<0$ and $n^{2}-n-365>0$, met only at $\boxed{n=20}$ — so
stand **20th** in line.

In [19]:
from fractions import Fraction

def birthday_line_prob(n, days=365):
    """P(the person at position n wins): the first n-1 people all have distinct birthdays and yours
    matches one of them."""
    if n < 2:
        return Fraction(0)
    distinct = Fraction(1)
    for k in range(n - 1):
        distinct *= Fraction(days - k, days)
    return distinct * Fraction(n - 1, days)

best = max(range(2, 100), key=birthday_line_prob)
print("best position:", best, " with P =", float(birthday_line_prob(best)))
for n in (19, 20, 21):
    print(f"  n={n}: P = {float(birthday_line_prob(n)):.5f}")

best position: 20  with P = 0.0323198575490433
  n=19: P = 0.03221
  n=20: P = 0.03232
  n=21: P = 0.03225


### 4.3.8 Dice order

**Problem.** Three dice are thrown one by one. What is the probability the three values come out in
**strictly increasing** order?

**Logic (condition on distinctness).** Strictly increasing forces all three to differ, so split it:
$$P=\underbrace{P(\text{all different})}_{1\cdot\frac56\cdot\frac46=\frac{5}{9}}\times
\underbrace{P(\text{increasing}\mid\text{all different})}_{1/3!=\frac16}=\frac59\cdot\frac16=\boxed{\frac{5}{54}}.$$
Given three distinct values exactly one of their $3!$ orderings is increasing, hence the $\tfrac16$.
Equivalently, choose $3$ distinct faces and lay them in the one increasing order:
$P=\binom{6}{3}/6^{3}=20/216=5/54$.

In [20]:
from math import comb
from fractions import Fraction

def dice_increasing_prob(k=3, sides=6):
    """P(k dice thrown in sequence come out strictly increasing) = C(sides, k) / sides**k
    (choose k distinct faces; exactly one of their orderings is increasing)."""
    return Fraction(comb(sides, k), sides ** k)

print("3 dice, 6 sides:", dice_increasing_prob(), "=", float(dice_increasing_prob()))
for k, s in [(3, 6), (2, 6), (3, 20)]:
    print(f"  k={k}, sides={s}: {dice_increasing_prob(k, s)}")

3 dice, 6 sides: 5/54 = 0.09259259259259259
  k=3, sides=6: 5/54
  k=2, sides=6: 5/12
  k=3, sides=20: 57/400


### 4.3.9 Monty Hall problem

**Problem.** Three doors: one hides a car, two hide goats. You pick a door; the host — who knows the
layout — opens a *different* door revealing a goat, then offers you the switch. Should you switch, and
with what probability do you win?

**Logic (switching wins exactly when your first pick was wrong).** Your initial guess is the car with
probability $\tfrac13$ and a goat with probability $\tfrac23$. If you **switch**: whenever you started on
a goat (prob $\tfrac23$) the host is forced to reveal the *other* goat, so the remaining door is the car —
you win; whenever you started on the car (prob $\tfrac13$) switching loses. Hence
$$P(\text{win}\mid\text{switch})=\boxed{\tfrac23},\qquad P(\text{win}\mid\text{stay})=\tfrac13.$$
**Switch.** The flaw in the "it's now 50/50" argument is that the host's reveal is *not* independent of
where the car is — it leaks information. (With $N$ doors and one goat opened, switching to a random other
door wins with probability $\tfrac{N-1}{N}\cdot\tfrac1{N-2}$.)

In [21]:
import random

def monty_hall_sim(trials=200000, doors=3, seed=0):
    """Monte-Carlo win rates for switching vs staying: the host opens a goat door different from your
    pick, and you switch to a random remaining door."""
    rng = random.Random(seed)
    switch_wins = stay_wins = 0
    for _ in range(trials):
        car = rng.randrange(doors)
        pick = rng.randrange(doors)
        host = rng.choice([d for d in range(doors) if d != pick and d != car])
        newpick = rng.choice([d for d in range(doors) if d != pick and d != host])
        switch_wins += (newpick == car)
        stay_wins += (pick == car)
    return switch_wins / trials, stay_wins / trials

sw, st = monty_hall_sim()
print(f"switch wins ~ {sw:.3f}  (= 2/3),   stay wins ~ {st:.3f}  (= 1/3)")

switch wins ~ 0.666  (= 2/3),   stay wins ~ 0.334  (= 1/3)


### 4.3.10 Amoeba population

**Problem.** A pond starts with one amoeba. Each minute every amoeba independently **dies**, **stays the
same**, **splits into two**, or **splits into three**, each with probability $\tfrac14$; offspring behave
the same way. What is the probability the population eventually dies out?

**Logic (condition on the first minute).** Let $P$ be the extinction probability starting from one amoeba.
By the law of total probability over the four equally likely first moves — and because distinct amoeba
lines die out independently (two lines die with probability $P^{2}$, three with $P^{3}$):
$$P=\tfrac14\cdot1+\tfrac14\,P+\tfrac14\,P^{2}+\tfrac14\,P^{3}.$$
Multiplying by $4$ gives $P^{3}+P^{2}-3P+1=0$, which factors as $(P-1)(P^{2}+2P-1)=0$. The roots are
$1,\ -1-\sqrt2,\ -1+\sqrt2$; the only one in $(0,1)$ is
$$P=\sqrt2-1\approx\boxed{0.414}.$$
(The root $P=1$ is rejected: the mean offspring count is $\tfrac14(0+1+2+3)=\tfrac32>1$, so the colony is
*supercritical* and has a genuine chance to survive forever.)

In [22]:
def extinction_prob(offspring, iters=2000):
    """Extinction probability of a branching process: the smallest fixed point of the offspring
    generating function P = sum_j p_j P^j, found by iterating from P=0. `offspring` maps
    (# children) -> probability."""
    P = 0.0
    for _ in range(iters):
        P = sum(p * P ** j for j, p in offspring.items())
    return P

amoeba = {0: 0.25, 1: 0.25, 2: 0.25, 3: 0.25}
P = extinction_prob(amoeba)
print(f"extinction probability = {P:.6f}   (sqrt(2) - 1 = {2 ** 0.5 - 1:.6f})")
print("satisfies P^3 + P^2 - 3P + 1 = 0 ->", round(P ** 3 + P ** 2 - 3 * P + 1, 10))

extinction probability = 0.414214   (sqrt(2) - 1 = 0.414214)
satisfies P^3 + P^2 - 3P + 1 = 0 -> 0.0


### 4.3.11 Candies in a jar

**Problem.** A jar holds $10$ red, $20$ blue, and $30$ green candies, drawn out one by one. What is the
probability that when the **last red** is drawn there is **still at least one blue and one green** left?

**Logic (which colour finishes first).** "A blue and a green remain when the reds run out" means red is the
**first colour exhausted** — its last candy precedes the last blue *and* the last green. With $T_r,T_b,T_g$
the positions of each colour's last candy, we want $P(T_r<T_b\text{ and }T_r<T_g)$. Condition on the very
last candy overall (colour $c$ is last with probability $\text{count}_c/60$):
- **green last** (prob $\tfrac{30}{60}$): among the $30$ red+blue candies the last must be blue, prob $\tfrac{20}{30}$;
- **blue last** (prob $\tfrac{20}{60}$): among the $40$ red+green candies the last must be green, prob $\tfrac{30}{40}$.

(Red cannot be last, since it must finish first.) So
$$P=\frac{30}{60}\cdot\frac{20}{30}+\frac{20}{60}\cdot\frac{30}{40}=\frac13+\frac14=\boxed{\frac{7}{12}}.$$

In [23]:
from fractions import Fraction

def candies_prob(red, blue, green):
    """P(at least one blue and one green remain when the last red is drawn) = P(red is exhausted first).
    Condition on the colour of the very last candy (which must be blue or green)."""
    N = red + blue + green
    return (Fraction(green, N) * Fraction(blue, red + blue)
            + Fraction(blue, N) * Fraction(green, red + green))

p = candies_prob(10, 20, 30)
print(f"P(a blue and a green remain when the reds run out) = {p} = {float(p):.4f}")

P(a blue and a green remain when the reds run out) = 7/12 = 0.5833


### 4.3.12 Coin toss game

**Problem.** Players $A$ and $B$ alternately toss a fair coin ($A$ first, then $B$, then $A$, …). The game
ends the first time a head is **immediately followed by a tail** ($HT$), and the player who tossed that
**tail** wins. What is $P(A\text{ wins})$?

**Logic (condition on $A$'s first toss, then use symmetry).** Write $P=P(A\text{ wins})$.
- If $A$'s first toss is **$T$**, it can never be the $H$ of an $HT$, and $B$ is now effectively the first
  tosser — so $A$ sits in $B$'s old role: $P(A\mid T)=1-P$.
- If $A$'s first toss is **$H$**, condition on $B$'s toss: with prob $\tfrac12$ $B$ throws $T$ (completing
  $HT$ — $B$ wins, $A$ gets $0$); with prob $\tfrac12$ $B$ throws $H$ and becomes the new "standing $H$",
  putting $A$ in the responder's seat, so $P(A\mid H)=\tfrac12\cdot0+\tfrac12\big(1-P(A\mid H)\big)\Rightarrow P(A\mid H)=\tfrac13$.

Combining, $P=\tfrac12P(A\mid H)+\tfrac12P(A\mid T)=\tfrac12\cdot\tfrac13+\tfrac12(1-P)$, hence
$\tfrac32P=\tfrac23$ and
$$P(A\text{ wins})=\boxed{\tfrac49},\qquad P(B\text{ wins})=\tfrac59.$$
Reasonable: $A$ cannot possibly win on his first toss, yet $B$ already has a $\tfrac14$ chance to win on
her first, so $A$ is the underdog.

In [24]:
import random

def coin_toss_game_sim(trials=500000, seed=0):
    """Simulate A,B,A,B,... tossing a fair coin; the game ends at the first HT and the tosser of that T
    wins. A tosses on odd turns, B on even. Returns the empirical P(A wins) (~ 4/9)."""
    rng = random.Random(seed)
    a_wins = 0
    for _ in range(trials):
        prev, t = None, 0
        while True:
            t += 1
            toss = "H" if rng.random() < 0.5 else "T"
            if prev == "H" and toss == "T":
                a_wins += (t % 2 == 1)      # the tosser of turn t (A on odd t) tossed the winning T
                break
            prev = toss
    return a_wins / trials

print(f"simulated P(A wins) = {coin_toss_game_sim():.4f}   (exact 4/9 = {4/9:.4f})")

simulated P(A wins) = 0.4447   (exact 4/9 = 0.4444)


### 4.3.13 Aces

**Problem.** A $52$-card deck is dealt to $4$ players, $13$ cards each. What is the probability that
**every** player gets exactly one ace?

**Logic (place the aces one at a time).** Track where the four aces land. The first ace can go anywhere.
The second must avoid the first ace's hand — $39$ of the remaining $51$ slots lie in other hands. The
third must avoid both — $26$ of $50$; the fourth — $13$ of $49$:
$$P=\frac{52}{52}\cdot\frac{39}{51}\cdot\frac{26}{50}\cdot\frac{13}{49}
=\frac{39\cdot26\cdot13}{51\cdot50\cdot49}=\frac{2197}{20825}\approx\boxed{0.1055}.$$
In general, dealing $m\cdot c$ cards to $m$ players ($c$ each) with $m$ special cards, the chance each hand
holds exactly one is $\displaystyle\prod_{j=1}^{m}\frac{(m-j+1)\,c}{mc-j+1}$.

In [25]:
from fractions import Fraction

def aces_prob(players=4, per_hand=13):
    """P(each of `players` hands gets exactly one of the `players` aces) when m*c cards are dealt c each.
    Place the aces one by one, each avoiding the hands that already hold an ace."""
    N = players * per_hand
    p = Fraction(1)
    for j in range(1, players + 1):
        p *= Fraction((players - j + 1) * per_hand, N - j + 1)
    return p

print("standard deck:", aces_prob(), "=", float(aces_prob()))
for m, c in [(4, 13), (2, 5), (3, 4)]:
    print(f"  {m} players x {c} cards: {aces_prob(m, c)} = {float(aces_prob(m, c)):.4f}")

standard deck: 2197/20825 = 0.10549819927971188
  4 players x 13 cards: 2197/20825 = 0.1055
  2 players x 5 cards: 5/9 = 0.5556
  3 players x 4 cards: 16/55 = 0.2909


### 4.3.14 Gambler's ruin

**Problem.** A gambler starts with $\$i$ and, each game, wins $\$1$ with probability $p$ or loses $\$1$
with probability $q=1-p$, stopping at $\$0$ (ruin) or $\$N$ (target). What is $P_i$, the probability of
reaching $\$N$ before going broke?

**Logic (a boundary-value recurrence).** Conditioning on the next game gives $P_i=pP_{i+1}+qP_{i-1}$ with
$P_0=0$ and $P_N=1$. This linear recurrence has ratio $r=q/p$; solving with those boundaries,
$$P_i=\begin{cases}\dfrac{1-(q/p)^{\,i}}{1-(q/p)^{\,N}}, & p\neq\tfrac12,\\[2mm]\dfrac{i}{N}, & p=\tfrac12.\end{cases}$$
For a fair game the chance is just your stake's share, $i/N$; for an unfavourable game ($p<\tfrac12$, so
$r>1$) the ruin probability climbs steeply as the target $N$ grows — the arithmetic behind "the house
always wins.\"

In [26]:
from fractions import Fraction

def gamblers_ruin(i, N, p=Fraction(1, 2)):
    """P(reach N before 0) starting from i, winning each step with probability p. Fair game -> i/N;
    otherwise (1 - r^i)/(1 - r^N) with r = q/p."""
    p = Fraction(p).limit_denominator(10 ** 9)
    if p == Fraction(1, 2):
        return Fraction(i, N)
    r = (1 - p) / p
    return (1 - r ** i) / (1 - r ** N)

for p in [Fraction(1, 2), Fraction(49, 100), Fraction(2, 5)]:
    v = gamblers_ruin(5, 10, p)
    print(f"i=5, N=10, p={float(p)}: P(reach 10 before 0) = {v} = {float(v):.4f}")

i=5, N=10, p=0.5: P(reach 10 before 0) = 1/2 = 0.5000
i=5, N=10, p=0.49: P(reach 10 before 0) = 282475249/627500500 = 0.4502
i=5, N=10, p=0.4: P(reach 10 before 0) = 32/275 = 0.1164


### 4.3.15 Basketball scores

**Problem.** A player scores her first free throw and misses her second. Thereafter the probability she
scores throw $n+1$ equals the **fraction she has made so far**, $\text{made}/n$. After $100$ throws, what
is the probability she has made **exactly $50$**?

**Logic (Pólya-urn induction).** Let $P(n,k)$ be the chance of $k$ makes after $n$ throws. From $n=3$ one
gets $P(3,1)=P(3,2)=\tfrac12$, and the induction step via the law of total probability,
$$P(n{+}1,k)=\Big(1-\tfrac{k}{n}\Big)\tfrac{1}{n-1}+\tfrac{k-1}{n}\cdot\tfrac{1}{n-1}=\tfrac1n,$$
shows the made-count is **uniform**: $P(n,k)=\dfrac{1}{n-1}$ for every $k\in\{1,\dots,n-1\}$. Hence
$$P(100,50)=\boxed{\tfrac{1}{99}}.$$
The specific target $50$ is a red herring — every achievable count $1$–$99$ is equally likely.

In [27]:
from fractions import Fraction

def basketball_prob(n, k):
    """P(exactly k baskets after n throws) in the 'score-fraction' free-throw model: the made-count is
    uniform, so 1/(n-1) for any k in 1..n-1, else 0."""
    return Fraction(1, n - 1) if 1 <= k <= n - 1 else Fraction(0)

print("P(100 throws, exactly 50 made) =", basketball_prob(100, 50), "=", float(basketball_prob(100, 50)))
print("uniform over made-counts -> P(100,k) for k = 1, 50, 99:",
      basketball_prob(100, 1), basketball_prob(100, 50), basketball_prob(100, 99))

P(100 throws, exactly 50 made) = 1/99 = 0.010101010101010102
uniform over made-counts -> P(100,k) for k = 1, 50, 99: 1/99 1/99 1/99


### 4.3.16 Cars on road

**Problem.** The probability of seeing at least one car on a highway in any $20$-minute interval is
$\tfrac{609}{625}$. If sightings occur at a constant rate, what is the probability of seeing at least one
car in a given $5$-minute interval?

**Logic (independent sub-intervals + complement).** Split the $20$ minutes into $4$ disjoint, independent
$5$-minute intervals, each with the same $P(\text{no car})=1-p$. Then
$$(1-p)^{4}=P(\text{no car in }20\text{ min})=1-\tfrac{609}{625}=\tfrac{16}{625}.$$
Taking fourth roots, $1-p=\big(\tfrac{16}{625}\big)^{1/4}=\tfrac{2}{5}$ (since $16=2^{4}$, $625=5^{4}$), so
$$p=\boxed{\tfrac{3}{5}}.$$
General move: if an event has probability $P_T$ over a window split into $m$ equal independent pieces, each
piece carries probability $1-(1-P_T)^{1/m}$.

In [28]:
from fractions import Fraction

def _int_root(x, m):
    """Exact m-th root of a non-negative integer if it is a perfect m-th power, else None."""
    if x < 0:
        return None
    k = round(x ** (1 / m))
    for c in (k - 1, k, k + 1):
        if c >= 0 and c ** m == x:
            return c
    return None

def subinterval_prob(p_big=Fraction(609, 625), pieces=4):
    """Given P(event) = p_big over a window split into `pieces` equal independent sub-windows, return the
    per-sub-window probability 1 - (1-p_big)^(1/pieces) (exact when the root is rational)."""
    no = 1 - p_big
    rn, rd = _int_root(no.numerator, pieces), _int_root(no.denominator, pieces)
    if rn is not None and rd is not None:
        return 1 - Fraction(rn, rd)
    return 1 - float(no) ** (1 / pieces)

p = subinterval_prob()
print("P(at least one car in 5 min) =", p, "=", float(p))

P(at least one car in 5 min) = 3/5 = 0.6


## 4.4 Discrete and Continuous Distributions

A **random variable** $X$ attaches a number to each outcome. Everything you compute about it flows from a
handful of standard functions — **summed** for a *discrete* $X$, **integrated** for a *continuous* $X$:

| quantity | discrete | continuous |
|---|---|---|
| **CDF** $F(a)$ | $P(X\le a)$ | $\displaystyle\int_{-\infty}^{a} f(x)\,dx$ |
| **pmf / pdf** | $p(x)=P(X=x)$ | $f(x)=\dfrac{d}{dx}F(x)$ |
| **mean** $E[X]$ | $\displaystyle\sum_{x} x\,p(x)$ | $\displaystyle\int_{-\infty}^{\infty} x\,f(x)\,dx$ |
| $E[g(X)]$ | $\displaystyle\sum_{x} g(x)\,p(x)$ | $\displaystyle\int_{-\infty}^{\infty} g(x)\,f(x)\,dx$ |
| **variance** $\operatorname{Var}(X)$ | $E\big[(X-E[X])^{2}\big]=E[X^{2}]-E[X]^{2}$ | (same formula) |
| **std** $\operatorname{std}(X)$ | $\sqrt{\operatorname{Var}(X)}$ | (same) |

For a continuous $X$, $P(X=x)=0$, so $P(X\le a)=P(X<a)$: the pdf gives *densities*, not probabilities —
only integrals over intervals are probabilities. The next two tables collect the distributions worth
memorising, each with a one-line use case.

### Discrete distributions

| name | pmf $p(x)$ | $E[X]$ | $\operatorname{Var}(X)$ | models… |
|---|---|---|---|---|
| **Uniform** $\{a,\dots,b\}$ | $\dfrac{1}{b-a+1}$ | $\dfrac{a+b}{2}$ | $\dfrac{(b-a+1)^{2}-1}{12}$ | equally likely labels (a fair die) |
| **Binomial** $(n,p)$ | $\binom{n}{x}p^{x}(1-p)^{n-x}$ | $np$ | $np(1-p)$ | # successes in $n$ independent trials |
| **Poisson** $(\lambda t)$ | $\dfrac{e^{-\lambda t}(\lambda t)^{x}}{x!}$ | $\lambda t$ | $\lambda t$ | # rare events in a fixed window |
| **Geometric** $(p)$ | $(1-p)^{x-1}p$ | $\dfrac1p$ | $\dfrac{1-p}{p^{2}}$ | trial of the **first** success |
| **Neg. binomial** $(r,p)$ | $\binom{x-1}{r-1}p^{r}(1-p)^{x-r}$ | $\dfrac rp$ | $\dfrac{r(1-p)}{p^{2}}$ | trial of the **$r$-th** success |

**Gacha reading** (rate-up SSR probability $p=0.75\%$ per pull). One pull is a **categorical** draw —
R $79\%$, SR $18\%$, off-banner SSR $2.25\%$, rate-up SSR $0.75\%$. Then: the number of rate-up SSRs in
$n$ pulls is $\mathrm{Bin}(n,0.0075)$ (**binomial**); the pull on which your *first* rate-up SSR lands is
$\mathrm{Geom}(0.0075)$, mean $1/0.0075\approx133$ (**geometric**); the pull on which your **5th** lands
is $\mathrm{NB}(5,0.0075)$ (**negative binomial**, the worked example below); and over many pulls the
count is well-approximated by $\mathrm{Poisson}(\lambda)$ with $\lambda=0.0075\,n$.

### Continuous distributions

| name | pdf $f(x)$ | $E[X]$ | $\operatorname{Var}(X)$ | models… |
|---|---|---|---|---|
| **Uniform** $[a,b]$ | $\dfrac{1}{b-a}$ | $\dfrac{a+b}{2}$ | $\dfrac{(b-a)^{2}}{12}$ | "no information" over a range |
| **Normal** $(\mu,\sigma^{2})$ | $\dfrac{1}{\sqrt{2\pi}\,\sigma}\,e^{-(x-\mu)^{2}/2\sigma^{2}}$ | $\mu$ | $\sigma^{2}$ | sums & averages via the CLT; asset returns |
| **Exponential** $(\lambda)$ | $\lambda e^{-\lambda x}$ | $\dfrac1\lambda$ | $\dfrac1{\lambda^{2}}$ | waiting time at a constant rate (memoryless) |
| **Gamma** $(\alpha,\lambda)$ | $\dfrac{\lambda e^{-\lambda x}(\lambda x)^{\alpha-1}}{\Gamma(\alpha)}$ | $\dfrac\alpha\lambda$ | $\dfrac\alpha{\lambda^{2}}$ | wait until $\alpha$ events occur |
| **Beta** $(\alpha,\beta)$ | $\dfrac{\Gamma(\alpha+\beta)}{\Gamma(\alpha)\Gamma(\beta)}\,x^{\alpha-1}(1-x)^{\beta-1}$ | $\dfrac{\alpha}{\alpha+\beta}$ | $\dfrac{\alpha\beta}{(\alpha+\beta)^{2}(\alpha+\beta+1)}$ | an unknown probability / proportion on $[0,1]$ |

The **normal** dominates via the central limit theorem (sums of many small independent effects). The
**exponential** and **Poisson** are two views of one constant-rate process — gaps vs. counts — and the
**gamma** is a sum of exponentials (wait for the $\alpha$-th event), the continuous cousin of the
negative binomial. The **beta** lives on $[0,1]$, the natural prior for a *probability* — e.g. a Bayesian
estimate of a banner's true rate-up rate from observed pulls.

### Distributions in action — Umamusume gacha (solving for 5 rate-up SSRs)

**Setup.** Each pull independently yields a **rate-up SSR** with probability $p=0.0075$ ($0.75\%$).
"Getting 5 rate-up SSRs" is a **negative-binomial** question: the pull count $X$ on which the $5$th
rate-up SSR lands is $X\sim\mathrm{NB}(5,p)$, with $P(X=x)=\binom{x-1}{4}p^{5}(1-p)^{x-5}$.

**How many pulls to expect?** The mean is $E[X]=\dfrac{r}{p}=\dfrac{5}{0.0075}\approx\boxed{667\text{ pulls}}$
(standard deviation $\approx297$ — a *huge* spread). So it is more useful to ask how many pulls reach $5$
with a given confidence, i.e. the smallest $n$ with $P(\mathrm{Bin}(n,p)\ge5)\ge c$: about **623** pulls
for a coin-flip's chance, **1064** for $90\%$, and **1544** for $99\%$. Inside a fixed budget of $667$
pulls, $P(\ge5\text{ rate-up})\approx0.56$ and $P(\text{exactly }5)\approx0.18$ (matching a
$\mathrm{Poisson}(5)$ approximation, since $\lambda=0.0075\cdot667\approx5$). The code computes all of these.

In [29]:
from math import comb, exp, factorial

# Umamusume banner rates (per pull): R, SR, any SSR, rate-up SSR
P_R, P_SR, P_SSR, P_RATEUP = 0.79, 0.18, 0.03, 0.0075
assert abs(P_R + P_SR + (P_SSR - P_RATEUP) + P_RATEUP - 1) < 1e-9   # 79 + 18 + 2.25 + 0.75 = 100%

def binom_pmf(n, k, p):            # P(exactly k successes in n trials)
    return comb(n, k) * p ** k * (1 - p) ** (n - k)

def binom_at_least(n, k, p):       # P(>= k successes in n trials)
    return 1 - sum(binom_pmf(n, i, p) for i in range(k))

def negbin_pmf(x, r, p):           # P(the r-th success lands exactly on trial x)
    return comb(x - 1, r - 1) * p ** r * (1 - p) ** (x - r)

def pulls_for_confidence(r, p, conf):   # fewest pulls so P(>= r successes) >= conf
    n = r
    while binom_at_least(n, r, p) < conf:
        n += 1
    return n

r = 5
print(f"rate-up SSR probability p = {P_RATEUP}   (E[pulls to 1st] = 1/p = {1/P_RATEUP:.1f})")
print(f"E[pulls to {r} rate-up SSRs] = r/p = {r/P_RATEUP:.1f}   "
      f"(std = {(r*(1-P_RATEUP)/P_RATEUP**2)**0.5:.0f}, very wide)")
for c in (0.50, 0.90, 0.99):
    print(f"  fewest pulls for {c:.0%} chance of >= {r} rate-up SSRs: {pulls_for_confidence(r, P_RATEUP, c)}")
n = 667
print(f"in {n} pulls:  P(exactly 5) = {binom_pmf(n, 5, P_RATEUP):.4f}  "
      f"(Poisson(5) approx {exp(-5) * 5 ** 5 / factorial(5):.4f}),  P(>= 5) = {binom_at_least(n, 5, P_RATEUP):.4f}")

rate-up SSR probability p = 0.0075   (E[pulls to 1st] = 1/p = 133.3)
E[pulls to 5 rate-up SSRs] = r/p = 666.7   (std = 297, very wide)
  fewest pulls for 50% chance of >= 5 rate-up SSRs: 623
  fewest pulls for 90% chance of >= 5 rate-up SSRs: 1064
  fewest pulls for 99% chance of >= 5 rate-up SSRs: 1544
in 667 pulls:  P(exactly 5) = 0.1761  (Poisson(5) approx 0.1755),  P(>= 5) = 0.5606


### 4.4.1 Meeting probability

**Problem.** Two bankers each arrive at the station at a uniformly random time between 5:00 and 6:00,
independently; each waits exactly $5$ minutes then leaves. What is the probability they meet?

**Logic (geometric probability).** Let $X,Y\sim\mathrm{Uniform}[0,60]$ be their arrival minutes. They
overlap **iff** $|X-Y|\le5$. Plot $(X,Y)$ in the $60\times60$ square: the meeting region is the band around
the diagonal, and its complement is two right triangles with legs $55$, so
$$P=\frac{60^{2}-2\cdot\frac12\cdot55^{2}}{60^{2}}=\frac{3600-3025}{3600}=\frac{575}{3600}=\boxed{\frac{23}{144}}\approx0.16.$$
In general, over a window $[0,T]$ with wait $w$, $P=\dfrac{T^{2}-(T-w)^{2}}{T^{2}}=1-\big(1-\tfrac wT\big)^{2}$.

In [30]:
from fractions import Fraction
import random

def meeting_prob(T=60, w=5):
    """P(two independent Uniform[0,T] arrivals fall within w of each other) = (T^2 - (T-w)^2)/T^2."""
    return Fraction(T * T - (T - w) ** 2, T * T)

def meeting_sim(T=60, w=5, trials=500000, seed=0):
    rng = random.Random(seed)
    return sum(abs(rng.uniform(0, T) - rng.uniform(0, T)) <= w for _ in range(trials)) / trials

print("P(meet) =", meeting_prob(), "=", float(meeting_prob()), " | simulated:", round(meeting_sim(), 4))
for T, w in [(60, 5), (60, 10), (24, 1)]:
    print(f"  T={T}, w={w}: {meeting_prob(T, w)} = {float(meeting_prob(T, w)):.4f}")

P(meet) = 23/144 = 0.1597222222222222  | simulated: 0.1601
  T=60, w=5: 23/144 = 0.1597
  T=60, w=10: 11/36 = 0.3056
  T=24, w=1: 47/576 = 0.0816


### 4.4.2 Probability of triangle

**Problem.** A stick of length $1$ is cut at two independent uniform points. What is the probability the
three pieces can form a **triangle**?

**Logic (geometric probability).** Let the cuts be $X,Y\sim\mathrm{Uniform}[0,1]$. Three lengths form a
triangle **iff every piece is shorter than $\tfrac12$** (each less than the sum of the other two). In the
case $X<Y$ the pieces are $X,\ Y-X,\ 1-Y$, and the triangle inequalities become
$$Y>\tfrac12,\qquad Y<\tfrac12+X,\qquad X<\tfrac12,$$
a triangle of area $\tfrac18$ in the unit square; the symmetric case $X>Y$ adds another $\tfrac18$. Hence
$$P=\tfrac18+\tfrac18=\boxed{\tfrac14}.$$

In [31]:
import random

def triangle_sim(trials=500000, seed=0):
    """P(the 3 pieces of a unit stick cut at two uniform points form a triangle): every piece < 1/2."""
    rng = random.Random(seed)
    ok = 0
    for _ in range(trials):
        a, b = sorted((rng.random(), rng.random()))
        ok += max(a, b - a, 1 - b) < 0.5
    return ok / trials

print(f"P(triangle) simulated = {triangle_sim():.4f}   (exact 1/4 = 0.25)")

P(triangle) simulated = 0.2499   (exact 1/4 = 0.25)


### 4.4.3 Property of Poisson process

**Problem.** Buses arrive as a Poisson process with mean gap $1/\lambda=10$ min. You show up at a random
time. What is your expected wait for the next bus, and — on average — how long ago did the last bus leave?

**Logic (memorylessness).** Gaps between Poisson arrivals are $\mathrm{Exponential}(\lambda)$, which is
**memoryless**: $P(\tau>s+t\mid\tau>s)=P(\tau>t)$. The time already elapsed since the last bus tells you
nothing, so the wait for the **next** bus is still $\mathrm{Exp}(\lambda)$, with mean
$$E[\text{wait}]=\tfrac1\lambda=\boxed{10\text{ min}}.$$
By the same time-reversal symmetry, the last bus left $10$ minutes ago on average — so the gap you landed
in averages $10+10=\mathbf{20}$ minutes, **twice** the mean gap.

**Why the paradox (length-biased sampling).** Arriving at a random *time* makes you more likely to fall
inside a **long** gap than a short one, so the gap containing you is not a typical $\mathrm{Exp}(\lambda)$
gap. For a general interarrival $X$ the expected residual wait is
$$E[\text{residual}]=\frac{E[X^{2}]}{2\,E[X]},$$
which for the exponential ($E[X^{2}]=2/\lambda^{2}$, $E[X]=1/\lambda$) is $1/\lambda=10$ — while the whole
gap you occupy averages $E[X^{2}]/E[X]=2/\lambda=20$.

In [32]:
import random, bisect

def poisson_waiting_sim(mean_gap=10, horizon=400000, observations=200000, seed=0):
    """Simulate a Poisson bus process; sample random observation times and measure the forward wait, the
    time since the last bus, and the length of the gap you land in."""
    rng = random.Random(seed)
    arrivals, t = [], 0.0
    while t < horizon:
        t += rng.expovariate(1 / mean_gap)
        arrivals.append(t)
    fwd = bwd = length = 0.0
    for _ in range(observations):
        s = rng.uniform(mean_gap * 5, horizon - mean_gap * 5)
        i = bisect.bisect_left(arrivals, s)
        fwd += arrivals[i] - s
        bwd += s - arrivals[i - 1]
        length += arrivals[i] - arrivals[i - 1]
    n = observations
    return fwd / n, bwd / n, length / n

f, b, L = poisson_waiting_sim()
print(f"forward wait ~ {f:.2f} min        (= 1/lambda = 10)")
print(f"time since last bus ~ {b:.2f} min (= 10)")
print(f"gap you land in ~ {L:.2f} min     (= 20, the inspection paradox)")

forward wait ~ 10.01 min        (= 1/lambda = 10)
time since last bus ~ 9.97 min (= 10)
gap you land in ~ 19.97 min     (= 20, the inspection paradox)


### 4.4.4 Moments of the normal distribution

**Problem.** For $X\sim N(0,1)$, find $E[X^{n}]$ for $n=1,2,3,4$.

**Logic (moment generating function).** By symmetry every **odd** moment vanishes. For the rest, the MGF
$$M(t)=E[e^{tX}]=\int_{-\infty}^{\infty}e^{tx}\frac{1}{\sqrt{2\pi}}e^{-x^{2}/2}\,dx=e^{t^{2}/2}$$
packages all the moments, because $M^{(n)}(0)=E[X^{n}]$. Differentiating $e^{t^{2}/2}$:
$$M'(t)=t\,e^{t^{2}/2},\ \ M''(t)=(1+t^{2})e^{t^{2}/2},\ \ M'''(t)=(3t+t^{3})e^{t^{2}/2},\ \ M^{(4)}(t)=(3+6t^{2}+t^{4})e^{t^{2}/2}.$$
Evaluating at $t=0$,
$$E[X]=0,\quad E[X^{2}]=1,\quad E[X^{3}]=0,\quad E[X^{4}]=\boxed{3}.$$
These are the mean, variance, (zero) skewness, and kurtosis $3$ of the standard normal. In general
$E[X^{n}]=0$ for odd $n$ and the **double factorial** $(n-1)!!$ for even $n$ (so $E[X^{6}]=15$).

In [33]:
import random

def normal_moment(n):
    """E[X^n] for X ~ N(0,1): 0 for odd n, the double factorial (n-1)!! for even n."""
    if n % 2:
        return 0
    m, r = n - 1, 1
    while m > 1:
        r *= m
        m -= 2
    return r

def normal_moment_sim(n, trials=2000000, seed=0):
    rng = random.Random(seed)
    return sum(rng.gauss(0, 1) ** n for _ in range(trials)) / trials

for n in range(1, 5):
    print(f"E[X^{n}] = {normal_moment(n)}    (Monte Carlo {normal_moment_sim(n):+.3f})")
print("higher even moments (n-1)!!:", {n: normal_moment(n) for n in (6, 8, 10)})

E[X^1] = 0    (Monte Carlo +0.001)
E[X^2] = 1    (Monte Carlo +1.000)
E[X^3] = 0    (Monte Carlo +0.002)
E[X^4] = 3    (Monte Carlo +2.998)
higher even moments (n-1)!!: {6: 15, 8: 105, 10: 945}


## 4.5 Expected Value, Variance and Covariance

Expectation $E[X]$ and variance $\operatorname{Var}(X)=E[X^{2}]-E[X]^{2}$ came up in §4.4; this section adds
the tools for **two or more** variables. Throughout, a fair die $X$ (with $E[X]=3.5$ and
$\operatorname{Var}(X)=\tfrac{6^{2}-1}{12}=\tfrac{35}{12}$) is the running example.

### Covariance and correlation
**Covariance** measures how two variables move *together*:
$$\operatorname{Cov}(X,Y)=E\big[(X-E[X])(Y-E[Y])\big]=E[XY]-E[X]E[Y].$$
Positive means they tend to rise together, negative means one rises as the other falls, and independent
variables have $\operatorname{Cov}=0$ (though the converse is false). **Correlation** rescales it to
$[-1,1]$:
$$\rho(X,Y)=\frac{\operatorname{Cov}(X,Y)}{\sqrt{\operatorname{Var}(X)\operatorname{Var}(Y)}}.$$
*Worked example.* Let $Y=7-X$ (a die and its opposite face). Then
$\operatorname{Cov}(X,7-X)=\operatorname{Cov}(X,7)-\operatorname{Cov}(X,X)=0-\operatorname{Var}(X)=-\tfrac{35}{12}$,
and $\rho=\dfrac{-35/12}{\sqrt{(35/12)(35/12)}}=-1$ — perfectly anti-correlated, as it must be for an exact
decreasing linear relationship.

### The algebra of variance and covariance
$\operatorname{Cov}$ is symmetric and **bilinear**, and variance is covariance with itself:
$$\operatorname{Cov}(X,X)=\operatorname{Var}(X),\qquad \operatorname{Cov}(X,c)=0,\qquad
\operatorname{Cov}(aX+bY,Z)=a\operatorname{Cov}(X,Z)+b\operatorname{Cov}(Y,Z).$$
The rules used constantly:
$$\operatorname{Var}(aX+b)=a^{2}\operatorname{Var}(X),\qquad
\operatorname{Var}(X+Y)=\operatorname{Var}(X)+\operatorname{Var}(Y)+2\operatorname{Cov}(X,Y),$$
$$\operatorname{Var}\Big(\textstyle\sum_i X_i\Big)=\sum_i\operatorname{Var}(X_i)+2\sum_{i<j}\operatorname{Cov}(X_i,X_j).$$
For **independent** variables the cross terms vanish, so variance is additive.
*Worked example.* Two independent dice sum to variance $\tfrac{35}{12}+\tfrac{35}{12}=\tfrac{35}{6}$;
doubling one die instead gives $\operatorname{Var}(2X)=4\cdot\tfrac{35}{12}=\tfrac{35}{3}$ — larger, since
scaling amplifies spread more than adding an independent copy.

### Conditional expectation and variance
Conditioning on $Y$ reweights $X$'s distribution to the world where $Y$ is known:
$$E[X\mid Y{=}y]=\sum_{x} x\,P(X{=}x\mid Y{=}y)\ \text{(discrete)},\qquad
E[X\mid Y{=}y]=\int x\,f_{X\mid Y}(x\mid y)\,dx\ \text{(continuous)},$$
and the **conditional variance** is $\operatorname{Var}(X\mid Y)=E[X^{2}\mid Y]-E[X\mid Y]^{2}$. Crucially,
$E[X\mid Y]$ is itself a **random variable** — a function of $Y$.
*Worked example.* For a die, condition on parity: $E[X\mid\text{even}]=\tfrac{2+4+6}{3}=4$ and
$E[X\mid\text{odd}]=\tfrac{1+3+5}{3}=3$, each with $\operatorname{Var}(X\mid\cdot)=\tfrac{8}{3}$.

### The law of total expectation (and variance)
Averaging the conditional expectation back over $Y$ recovers the whole — the **tower rule**:
$$E[X]=E\big[E[X\mid Y]\big]=\sum_{y} E[X\mid Y{=}y]\,P(Y{=}y).$$
Its variance partner splits total spread into within-group and between-group pieces:
$$\operatorname{Var}(X)=\underbrace{E\big[\operatorname{Var}(X\mid Y)\big]}_{\text{within}}
+\underbrace{\operatorname{Var}\big(E[X\mid Y]\big)}_{\text{between}}.$$
*Worked example.* For the die by parity: $E[X]=4\cdot\tfrac12+3\cdot\tfrac12=3.5$ ✓, and
$\operatorname{Var}(X)=\underbrace{\tfrac{8}{3}}_{E[\operatorname{Var}]}+\underbrace{\tfrac14}_{\operatorname{Var}(E)}=\tfrac{35}{12}$ ✓.
The tower rule is the workhorse behind the connecting-noodles and dice-game problems below.

### 4.5.1 Connecting noodles

**Problem.** A bowl holds $100$ noodles ($200$ free ends). Blindfolded, you repeatedly join two random free
ends until none remain. What is the **expected number of loops** formed?

**Logic (condition on the first join → recursion).** With $n$ noodles there are $2n$ ends; pick one, and its
partner is one of the other $2n-1$. Exactly one of those is the *other end of the same noodle* — which closes
a loop and leaves $n-1$ noodles — while the other $2n-2$ merge two noodles into one, also leaving $n-1$. So,
writing $f(n)$ for the loop count,
$$E[f(n)]=\tfrac{1}{2n-1}\big(1+E[f(n-1)]\big)+\tfrac{2n-2}{2n-1}E[f(n-1)]=E[f(n-1)]+\frac{1}{2n-1}.$$
With $E[f(1)]=1$ this telescopes to
$$E[f(n)]=\sum_{k=1}^{n}\frac{1}{2k-1}=1+\tfrac13+\tfrac15+\cdots+\tfrac{1}{2n-1}.$$
For $n=100$ that is $\approx\boxed{3.28}$ loops — the odd-reciprocal sum grows like $\tfrac12\ln n$, so even
$100$ noodles rarely make more than a handful.

In [34]:
from fractions import Fraction
import random

def noodle_circles_expected(n):
    """Expected number of loops from randomly joining the 2n ends of n noodles.
    E[f(n)] = E[f(n-1)] + 1/(2n-1) = sum_{k=1}^{n} 1/(2k-1)."""
    return sum(Fraction(1, 2 * k - 1) for k in range(1, n + 1))

def noodle_sim(n, trials=20000, seed=0):
    """Simulate the joining process and count loops."""
    rng = random.Random(seed)
    total = 0
    for _ in range(trials):
        partner = {}
        for i in range(n):
            partner[2 * i], partner[2 * i + 1] = 2 * i + 1, 2 * i     # the two ends of noodle i
        free, circles = list(range(2 * n)), 0
        while free:
            a = free.pop(rng.randrange(len(free)))
            b = free.pop(rng.randrange(len(free)))
            if partner[a] == b:
                circles += 1                                          # closed a loop
            else:
                ap, bp = partner[a], partner[b]                       # merge two strands
                partner[ap], partner[bp] = bp, ap
        total += circles
    return total / trials

for n in (1, 2, 3, 100):
    print(f"n={n:>3}: E[loops] = {float(noodle_circles_expected(n)):.4f}")
print("Monte Carlo (n=100):", round(noodle_sim(100), 3))

n=  1: E[loops] = 1.0000
n=  2: E[loops] = 1.3333
n=  3: E[loops] = 1.5333
n=100: E[loops] = 3.2843
Monte Carlo (n=100): 3.298


### 4.5.2 Optimal hedge ratio

**Problem.** You own one share of stock $A$ and hedge by shorting $h$ shares of $B$. Given return variances
$\sigma_A^{2},\sigma_B^{2}$ and correlation $\rho$, which $h$ minimises the hedged variance?

**Logic (minimise a quadratic in $h$).** By the variance rules,
$$\operatorname{Var}(r_A-h\,r_B)=\sigma_A^{2}-2h\rho\sigma_A\sigma_B+h^{2}\sigma_B^{2}.$$
Setting the derivative in $h$ to zero, $-2\rho\sigma_A\sigma_B+2h\sigma_B^{2}=0$, gives
$$\boxed{\,h^{*}=\rho\,\frac{\sigma_A}{\sigma_B}\,}$$
(a genuine minimum, since $\partial^{2}/\partial h^{2}=2\sigma_B^{2}>0$). The residual variance is
$\sigma_A^{2}(1-\rho^{2})$ — the hedge strips away the fraction $\rho^{2}$ of $A$'s variance, and a perfect
hedge ($|\rho|=1$) drives it to zero. This $h^{*}$ is exactly the least-squares slope of $A$ on $B$ (the
regression "beta").

In [35]:
def optimal_hedge_ratio(sigma_A, sigma_B, rho):
    """Shares of B to short per share of A to minimise Var(r_A - h r_B). Minimising the quadratic
    sigma_A^2 - 2 h rho sigma_A sigma_B + h^2 sigma_B^2 gives h* = rho * sigma_A / sigma_B, with residual
    (minimum) variance sigma_A^2 (1 - rho^2). Returns (h*, min_variance)."""
    return rho * sigma_A / sigma_B, sigma_A ** 2 * (1 - rho ** 2)

for sA, sB, r in [(0.20, 0.25, 0.8), (0.30, 0.30, 0.5), (0.20, 0.40, 1.0)]:
    h, v = optimal_hedge_ratio(sA, sB, r)
    print(f"sigma_A={sA}, sigma_B={sB}, rho={r}: h* = {h:.4f} shares,  min variance = {v:.5f}")

sigma_A=0.2, sigma_B=0.25, rho=0.8: h* = 0.6400 shares,  min variance = 0.01440
sigma_A=0.3, sigma_B=0.3, rho=0.5: h* = 0.5000 shares,  min variance = 0.06750
sigma_A=0.2, sigma_B=0.4, rho=1.0: h* = 0.5000 shares,  min variance = 0.00000


### 4.5.3 Dice game

**Problem.** You roll a die and are paid its face value. On a $4,5,$ or $6$ you may roll again; on a $1,2,$
or $3$ the game stops. What is the expected payoff?

**Logic (law of total expectation).** Let $E[X]$ be the expected payoff and condition on the first roll $Y$.
- With prob $\tfrac12$, $Y\in\{1,2,3\}$: you keep $Y$, worth $E[Y\mid Y\le3]=2$.
- With prob $\tfrac12$, $Y\in\{4,5,6\}$: you bank $E[Y\mid Y\ge4]=5$ **and** play again, adding another $E[X]$.

By the tower rule,
$$E[X]=\tfrac12\cdot2+\tfrac12\big(5+E[X]\big)\ \Longrightarrow\ \tfrac12E[X]=\tfrac72\ \Longrightarrow\ E[X]=\boxed{7}.$$
In general, re-rolling on a set $R$ of faces gives $E[X]=\dfrac{\sum_{\text{all faces}}f}{n-|R|}$ (here $21/3=7$).

In [36]:
from fractions import Fraction

def dice_game_expected(faces=tuple(range(1, 7)), reroll=(4, 5, 6)):
    """Expected payoff: roll a die, paid its face value, re-roll while the face is in `reroll`.
    E[X] = mean_face + (|reroll|/n) E[X]  =>  E[X] = sum(faces) / (n - |reroll|)."""
    return Fraction(sum(faces), len(faces) - len(reroll))

print("standard game (re-roll on 4,5,6):", dice_game_expected(), "=", float(dice_game_expected()))
print("re-roll on 6 only:", dice_game_expected(reroll=(6,)), "=", float(dice_game_expected(reroll=(6,))))

standard game (re-roll on 4,5,6): 7 = 7.0
re-roll on 6 only: 21/5 = 4.2


### 4.5.4 Card game

**Problem.** In a shuffled $52$-card deck, what is the expected number of cards you turn over to see the
**first ace**?

**Logic (indicators + linearity).** The four aces split the other $48$ cards into $5$ gaps. Let $X_i=1$ if
ordinary card $i$ appears **before all four aces**. Card $i$ and the four aces are in uniformly random
relative order, so $i$ is first among those five with probability $P(X_i=1)=\tfrac15$. The count up to and
including the first ace is $X=1+\sum_{i=1}^{48}X_i$, so by linearity
$$E[X]=1+\sum_{i=1}^{48}\tfrac15=1+\tfrac{48}{5}=\boxed{10.6}.$$
For $m$ ordinary and $n$ special cards, the same argument gives $E=1+\dfrac{m}{n+1}$.

In [37]:
from fractions import Fraction
import random

def first_special_position(ordinary=48, special=4):
    """Expected number of cards turned over to reach the first of `special` special cards among `ordinary`
    others (all orders equally likely). Each ordinary card precedes all specials with prob 1/(special+1),
    so E = 1 + ordinary/(special+1)."""
    return 1 + Fraction(ordinary, special + 1)

def first_ace_sim(ordinary=48, special=4, trials=200000, seed=0):
    rng = random.Random(seed)
    deck = ["A"] * special + ["x"] * ordinary
    total = 0
    for _ in range(trials):
        rng.shuffle(deck)
        total += next(i for i, c in enumerate(deck) if c == "A") + 1
    return total / trials

print("first ace in a 52-card deck:", first_special_position(), "=", float(first_special_position()),
      " | simulated:", round(first_ace_sim(), 3))

first ace in a 52-card deck: 53/5 = 10.6  | simulated: 10.596


### 4.5.5 Sum of random variables

**Problem.** Let $X_1,\dots,X_n$ be independent $\mathrm{Uniform}[0,1]$. What is $P(S_n\le1)$, where
$S_n=X_1+\cdots+X_n$?

**Logic (volume of a simplex).** The event $\{X_i\ge0,\ X_1+\cdots+X_n\le1\}$ is the standard
$n$-dimensional **simplex**, of volume $\tfrac{1}{n!}$. Because the $X_i$ are uniform on the unit cube
(density $1$), that volume *is* the probability:
$$P(S_n\le1)=\boxed{\frac{1}{n!}}.$$
Induction confirms it: $P(S_n\le1)=\int_0^1 P(S_{n-1}\le1-x)\,dx=\int_0^1\frac{(1-x)^{n-1}}{(n-1)!}\,dx=\frac{1}{n!}$,
from the base $P(S_1\le1)=1$. (Sanity: $n=2$ is a triangle of area $\tfrac12$, $n=3$ a tetrahedron of volume
$\tfrac16$.)

In [38]:
from math import factorial
from fractions import Fraction
import random

def simplex_prob(n):
    """P(X_1 + ... + X_n <= 1) for iid Uniform[0,1] = the volume of the standard simplex = 1/n!."""
    return Fraction(1, factorial(n))

def simplex_sim(n, trials=500000, seed=0):
    rng = random.Random(seed)
    return sum(sum(rng.random() for _ in range(n)) <= 1 for _ in range(trials)) / trials

for n in (1, 2, 3, 4, 5):
    line = f"n={n}: P(S_n <= 1) = {simplex_prob(n)} = {float(simplex_prob(n)):.5f}"
    if n <= 3:
        line += f"   (simulated {simplex_sim(n):.4f})"
    print(line)

n=1: P(S_n <= 1) = 1 = 1.00000   (simulated 1.0000)
n=2: P(S_n <= 1) = 1/2 = 0.50000   (simulated 0.5009)
n=3: P(S_n <= 1) = 1/6 = 0.16667   (simulated 0.1668)
n=4: P(S_n <= 1) = 1/24 = 0.04167
n=5: P(S_n <= 1) = 1/120 = 0.00833


---
*More of Chapter 4 as I keep reading.*